# cusmic demo [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/cusmic/blob/main/demo/demo.ipynb)

Select a GPU runtime.
This notebook uses the committed L.A.Cosmic reference.
To regenerate it separately, run `test/mkref.py OUTPUT` with L.A.Cosmic 1.4.0.
Locally, install the demo dependencies and open this file with JupyterLab.
The full image includes them; run it with `--gpus all -p 8888:8888` and arguments `-m jupyterlab --ip=0.0.0.0 --no-browser --allow-root`.

In [ ]:
import sys

revision = "main"  # Use one revision for code, reference files and benchmarks.

if "google.colab" in sys.modules:
    %pip install -q "cupy-cuda12x==14.2.0" "cuda-toolkit[cudart,nvrtc,cccl]==12.9.1" astropy click matplotlib
    %pip install -q --no-deps git+https://github.com/rndsrc/cusmic.git@{revision}

In [ ]:
from pathlib import Path
from urllib.request import urlopen

from cusmic.io import read_fits

root = Path.cwd()
if root.name == "demo":
    root = root.parent

base = f"https://raw.githubusercontent.com/rndsrc/cusmic/{revision}"
for name in ("test/data/input.fits.gz", "test/data/error.fits.gz",
             "test/data/reference.fits.gz", "bench/bench.py"):
    path = root / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(urlopen(f"{base}/{name}").read())

sys.path.insert(0, str(root))
data = root / "test/data"
image, _ = read_fits(data / "input.fits.gz", dtype="float64")
error, _ = read_fits(data / "error.fits.gz", dtype="float64")
reference, _ = read_fits(data / "reference.fits.gz", dtype="float64")
reference_mask, _ = read_fits(data / "reference.fits.gz", ext="CRMASK")

In [ ]:
import cupy as cp
import numpy as np
from cusmic import remove_cosmics

settings = {
    'contrast':1,
    'cr_threshold':5,
    'neighbor_threshold':5,
    'maxiter':4,
}

cleaned, mask = remove_cosmics(cp.asarray(image), error=cp.asarray(error), **settings)
cleaned, mask = cp.asnumpy(cleaned), cp.asnumpy(mask)

np.testing.assert_array_equal(cleaned.view("uint64"), reference.view("uint64"))
np.testing.assert_array_equal(mask, reference_mask)

print(f"Exact reference agreement; {mask.sum():,} cosmic-ray pixels detected.")

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, PowerNorm

low, high = np.percentile(image, (2, 99.7))
intensity = PowerNorm(0.5, vmin=low, vmax=high)
removed = image - cleaned
signal = Normalize(0, max(1, removed.max()))
difference = cleaned - reference
scale = max(1e-12, np.abs(difference).max())

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
panels = [
    (image, "Input", intensity, "gray"),
    (reference, "Cleaned reference", intensity, "gray"),
    (cleaned, "Cleaned cusmic", intensity, "gray"),
    (image - reference, "Removed signal: reference", signal, "gray"),
    (removed, "Removed signal: cusmic", signal, "gray"),
    (difference, "Error: cusmic - reference", Normalize(-scale, scale), "RdBu_r"),
]
for ax, (pixels, title, norm, cmap) in zip(axes.flat, panels):
    shown = ax.imshow(pixels, origin="lower", norm=norm, cmap=cmap)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(shown, ax=ax, shrink=0.7)

## Benchmark

The same benchmark runs from the command line.
Warmed timings wait for GPU completion and separate cleaning, transfers, and complete calls.
This notebook's first-result timing uses an already initialized GPU.

In [ ]:
from bench.bench import benchmark

record, _ = benchmark(image, error, settings, warmups=2, repeats=5)

record